In [1]:
#We import the necessary libraries and modules
import ast
import json
import numpy as np
from modules import spark_init
from modules import text_extractor as te
from modules import network_constructor as nc
from modules import demographic_network_constructor as dnc

#Paths where the data is stored
data_path = "data_shared/reddit/"

---
## **1. Attention-Flow Graph**

Here we construct the Attention-Flow Graph (AFG) using data from Reddit submisions from 2019-2023. We use a sample dataset containing $\sim 10\%$ percent of all the users that posted in Reddit during this time period.

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
#You need to save Pushshift data in the following format: data/reddit/submissions/{year}/RS_{year}-{month}.bz2
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]

#Here we read the data
processed_rdds = []
for timestep, path in enumerate(paths):
    rdd = sc.textFile(path)
    processed_rdd =  nc.data_preprocessing(rdd)
    attention_ratios = nc.get_attention_ratio(processed_rdd, timestep)
    processed_rdds.append(attention_ratios)

#Here we concatenate the RDDs
full_rdd = sc.union(processed_rdds)
#Here we save the RDD in a compressed format
full_rdd.saveAsTextFile(f"{data_path}/fraction_of_submissions_sampled", compressionCodecClass="org.apache.hadoop.io.compress.GzipCodec")
#Here we stop the Spark context
sc.stop()

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we read the data in an rdd
rdd = sc.textFile(f"{data_path}/fraction_of_submissions_sampled").coalesce(1000)

#Here we create the graph data
rescaled_temp_graph = nc.calculate_temporal_graph_data(rdd)

#Here we aggregate the graph
aggregated_graph = nc.calculate_attention_flow_graph(rescaled_temp_graph, total_months)

#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Total months: 60


Here we combine all the files generated by Spark in a single tsv file:

In [ ]:
! cat data_shared/reddit/attention_flow_graph_sampled/part-* > data_shared/supplementary_data/attention_flow_graph_sampled.tsv

---

## **2. Weight Rescaling**

Here we rescale the weights of the AFG using different methodologies (arithmetic, geometric mean). Then we compare the node strength against the subreddit popularity to assess if we effectively dampened the correlation between these two factors.

First we define some helping functions:

In [2]:
def resilient_eval(line:str):
    """
    This function evaluates a line.
    Args:
        line (str): The line.
    Returns
        ast.literal_eval(line) (tuple): The tuple.
    """
    try:
        return ast.literal_eval(line)
    except:
        return ()  
    
def obtain_examples_by_month(rdd, month:int):
    """
    This function obtains the examples by month.
    Args:
        rdd (RDD): The RDD.
        month (int): The month.
    Returns:
        (RDD): The RDD.
    """
    return (
        rdd
        .map(resilient_eval) # (subreddit_i, subreddit_j, timestep, weight, arithmetic_rescaled_weight, geometric_rescaled_weight)
        .filter(lambda x: x[2] == month) # We only consider the 10th month
        .map(lambda x: x[0]+","+x[1]+","+str(x[3])+","+str(x[4])+","+str(x[5])) # (subreddit_i, subreddit_j, weight, arithmetic_rescaled_weight, geometric_rescaled_weight)
    )

def monthly_authors(sc, timestep:int):
    """
    This function calculates the number of unique authors by month.
    Args:
        sc (SparkContext): The Spark context.
        timestep (int): The timestep.
    Returns:
        (RDD): The RDD.
    """
    min_year = 2019
    max_year = 2023

    #Here we define the paths to the data
    years = [str(year) for year in range(min_year, max_year+1)]
    months = [str(month).zfill(2) for month in range(1, 13)]
    paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]
    path = paths[timestep]
    rdd = sc.textFile(path)
    processed_rdd =  nc.data_preprocessing(rdd) #((author, subreddit), counts)
    #Now we get the number of unique authors
    unique_authors = processed_rdd.map(lambda x: (x[0][1],1)).reduceByKey(lambda x,y: x+y).map(lambda x: x[0]+","+str(x[1]))
    return unique_authors
    

Here we obtain some examples for the temporal graph:

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we read the data in an rdd
rdd = sc.textFile("data/temporal_graph_data_sampled").coalesce(1000)

#Here we create the graph data
example_flow_month_49 = obtain_examples_by_month(rdd, 49).saveAsTextFile(f"{data_path}/example_flow_month_49")

#Here we stop the Spark context
sc.stop()

Total months: 60


In [ ]:
! cat data_shared/reddit/example_flow_month_49/part-* > data_shared/supplementary_data/example_flow_month_49.tsv

We also obtain the number of monthly authors of each involved subreddit:

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we get the unique authors of month 29
unique_authors_month_29 = monthly_authors(sc, 49)
unique_authors_month_29.saveAsTextFile(f"{data_path}/unique_authors_month_49")

#Here we stop the Spark context
sc.stop()

Total months: 60


In [ ]:
! cat data_shared/reddit/unique_authors_month_49/part-* > data_shared/supplementary_data/unique_authors_month_49.tsv

---

## **3. User Scores**

Here we project the social scores for different demographic axes (age, gender, partisanship, affluence) defined by Waller and Anderson for each of the participating users as:

$$ F_u = \frac{\sum_s N_{u,s}F_s}{\sum_s N_{u,s}} $$

We project in two different ways: by taking into account the scores of the studied subreddits, and by ignoring them.

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]

#Here we read all the data in a single RDD
rdd = sc.textFile(','.join(paths))
#Here we obtain the user scores
processed_rdd = dnc.data_preprocessing(rdd)
user_scores = dnc.calculate_user_scores(processed_rdd)
#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [ ]:
! cat data_shared/reddit/normalized_demographic_scores/part-* > data_shared/reddit/normalized_demographic_scores.tsv

In [ ]:
! cat data_shared/reddit/normalized_sparse_demographic_scores/part-* > data_shared/reddit/normalized_sparse_demographic_scores.tsv

---

## **4. Demographic Attention-Flow**

Here we create the AFG by weighting the flow between subreddits by their projected score as:

$$ \mathbf{F}^{(t)} = \sum_{u\in \mathcal{U}} \mathbf{F}^{(t,u)} \cdot F_u$$

We do it for the both approaches mentioned before.


In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we read the data in an rdd
rdd = sc.textFile(f"{data_path}/fraction_of_submissions_sampled").coalesce(1000)

#Here we create the graph data
attention_flow = dnc.calculate_temporal_graph_data(rdd)
#We obtain the aggregated graph propagating scores
aggregate_by_user = dnc.aggregated_attention_flow(attention_flow, True)

#Here we aggregate the graphs
att_flow = dnc.calculate_attention_flow_graph(aggregate_by_user, total_months, True)

#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Total months: 60


In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we read the data in an rdd
rdd = sc.textFile(f"{data_path}/fraction_of_submissions_sampled").coalesce(1000)

#Here we create the graph data
attention_flow = dnc.calculate_temporal_graph_data(rdd)
#We obtain the aggregated graph propagating scores

aggregate_by_user = dnc.aggregated_attention_flow(attention_flow, False)

#Here we aggregate the graphs
att_flow = dnc.calculate_attention_flow_graph(aggregate_by_user, total_months, False)

#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Total months: 60


In [ ]:
! cat data_shared/reddit/attention_flow_graph_scores_sparse/part-* > data_shared/supplementary_data/attention_flow_graph_scores_sparse.tsv

---

## **5. URL Extraction**

Here we extract the URLs shared by all the subreddits taken into consideration since we want to link offline with online traits of these subreddits.

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12
print("Total months:", total_months)

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]

#Here we read the data
processed_rdds = []
for timestep, path in enumerate(paths):
    rdd = sc.textFile(path)
    processed_rdd =  te.data_preprocessing(rdd)
    processed_rdds.append(processed_rdd)
#Here we concatenate the RDDs
full_rdd = sc.union(processed_rdds)

#Now we count URL frequencies by subreddit
url_counts = te.count_urls(full_rdd)

#Now we save the url_counts in a json file such that the key is the subreddit and the value is a list of tuples (url, count)
json_dict_url = dict(url_counts.collect())
# Save as a single JSON file
with open(f"{data_path}/url_counts.json", "w") as f:
    json.dump(json_dict_url, f)

#Finally, we count youtube video_id frequencies by subreddit
video_id_counts = te.get_youtube_info(full_rdd)

#Now we save the video_id_counts in a json file such that the key is the subreddit and the value is a list of tuples (video_id, count)
json_dict_video = dict(video_id_counts.collect())
# Save as a single JSON file
with open(f"{data_path}/video_id_counts.json", "w") as f:
    json.dump(json_dict_video, f)

#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Total months: 60


---

## **7. Counting**

Here we perform a simple script to count the number of submissions and authors used for this study.

In [ ]:

def subreddit_monthly_stats(sc):
    """
    Returns for each subreddit:
      - the list of monthly unique author counts
      - median number of authors
      - average number of authors
    """
    min_year = 2019
    max_year = 2023
    years = [str(year) for year in range(min_year, max_year+1)]
    months = [str(month).zfill(2) for month in range(1, 13)]
    paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" 
             for year in years for month in months]

    monthly_rdds = []

    for path in paths:
        rdd = sc.textFile(path)
        processed_rdd = nc.data_preprocessing(rdd)  # ((author, subreddit), count)
        
        # Count unique authors per subreddit for this month
        authors_per_subreddit = processed_rdd.map(lambda x: (x[0][1], 1)) \
                                             .reduceByKey(lambda a, b: a + b)
        monthly_rdds.append(authors_per_subreddit)

    # Combine all months: (subreddit, count)
    combined_rdd = sc.union(monthly_rdds)

    # Group all monthly counts per subreddit into a list
    subreddit_monthly_list = combined_rdd.groupByKey() \
                                        .mapValues(lambda counts: list(counts))

    # Compute median and average
    def compute_stats(counts):
        counts_list = list(counts)
        return {
            "monthly_counts": counts_list,
            "median": float(np.median(counts_list)),
            "avg": float(np.mean(counts_list))
        }

    subreddit_stats = subreddit_monthly_list.mapValues(compute_stats)

    return subreddit_stats  # RDD: (subreddit, {"monthly_counts": [...], "median": x, "avg": y})

#Here we initialize the Spark context
sc = spark_init.spark_context()
stats_rdd = subreddit_monthly_stats(sc)
# Convert dicts to JSON strings
subreddit_stats_json = stats_rdd.map(lambda x: f"{x[0]}\t{json.dumps(x[1])}")
# Save the results
subreddit_stats_json.saveAsTextFile(f"{data_path}/subreddit_monthly_stats", compressionCodecClass="org.apache.hadoop.io.compress.GzipCodec")
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
